https://docs.sqlalchemy.org/en/20/tutorial/index.html#unified-tutorial

# SQLAlchemy Unified Tutorial

The SQLAlchemy Unified Tutorial is integrated between the Core and ORM components of SQLAlchemy and serves as a unified introduction to SQLAlchemy as a whole. For users of SQLAlchemy within the 1.x series, in the [2.0 style](https://docs.sqlalchemy.org/en/20/glossary.html#term-2.0-style) of working, the ORM uses Core-style querying with the [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) construct, and transactional semantics between Core connections and ORM sessions are equivalent. Tke note of the blue border styles for each section, that will tell you how "ORM-ish" a particular topic is!

Users who are already familiar with SQL Alchemy, and especially those looking to migrate existing applications to work under SQLAlchemy 2.0 series within the 14 transitional phase should check out the [SQLAlchemy 2.0 - Major Migration Guide](https://docs.sqlalchemy.org/en/20/changelog/migration_20.html) document as well.

For the newcomer, this document has a **lot** of detail, however by the end they will be considered an **Alchemist**

SQLAlchemy is presented as two distinct APIs, one building on top of the other. These APIs are known as **Core** and **ORM**.

**SQLAlchemy Core** is the foundational architecture for SQLAlchemy as a "database toolkit". The library provides tools for managing connectivity to a database, interacting with database queries and results, and programmatic construction of SQL statements.

Sections that are **primarily Core-only** will not refer to the ORM. SQLAlchemy constructs used in these sections will be imported from the `sqlalchemy` namespace. As an additional indicator of subject classification, they will also include a **dark blue border on the right**. When using the ORM, these concept are still in play but are less often explicit in user code. ORM users should read these sections, but not expect to be using these APIs directly for ORM-centric code.


**SQLAlchemy ORM** builds upon the Core to provide optional **object relational mapping** capabilities. The ORM provides an additional configuration layer allowing user-defined Python classes to be **mapped** to database tables and other constructs, as well as an object persistence mechanism known as the **Session**. It then extends the Core-level SQL Expression Language to allow SQL queries to be composed and invoked in terms of user-defined objects.

Sections that are **primarily ORM-only** should be **titled to include the phrase 'ORM'**, so that it's clear this is an ORM related topic. SQLAlchemy constructs used in these sections will be imported from the `sqlalchemy.orm` namespace. Finally, as an additional indicator of subject classification, they will also include a **light blue border on the left**. Core-only users can skip these.

**Most** sections in this tutorial discuss **Core concepts that are also used explicitly with the ORM**. SQLAlchemy 2.0 in particular features a much greater level of integration of Core API use within the ORM. For each of these sections, there will be **introductory text** discussing the degree to which ORM users should expect to be using these prorgamming patterns. SQLAlchmey constructs in these sections will be imported from the `sqlalchemy` namespace with some potential use of `sqlalchemy.orm` constructs at the same time. As an additional indicator of subject classification, these sections will also include **both a thinner light border on the left, and a thicker dark border on the right**. Core and ORM users should familiarize with concepts in these sections equally.


# Tutorial Overview

The tutorial will present both concepts in the natural order that they should be learned, first with a mostly-Core-centric approach and then spanning out into more ORM-centric concepts. The major sections of this tutorial are as follows:

* [Establishing Connectivity - the Engine](https://docs.sqlalchemy.org/en/20/tutorial/engine.html#tutorial-engine) - all SQLAlchemy applications start with an **Engine** object; here's how to create one.
* [Working with Transactions and the DBAPI](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-working-with-transactions) - the usage API of the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) and its related objects [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) and [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) are presented here. This content is Core-centric however ORM users will want to be familiar with at least the [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) obejct.
* [Working with Database Metadata](https://docs.sqlalchemy.org/en/20/tutorial/metadata.html#tutorial-working-with-metadata) - SQLAlchemy's SQL abstractions as well as the ORM rely upon a system of defining database schema consructs as Python objects. This section introduces how to do that from both a Core and an ORM perspective.
* [Working with Data](https://docs.sqlalchemy.org/en/20/tutorial/data.html#tutorial-working-with-data) - here we learn how to create, select, update and delete data in the database. The so-called [CRUD](https://docs.sqlalchemy.org/en/20/glossary.html#term-CRUD) operations here are given in terms of SQLAlchemy Core with links out towards their ORM counterparts. The SELECT operation that is introduced in detail at [Using SELECT Statements](https://docs.sqlalchemy.org/en/20/tutorial/data_select.html#tutorial-selecting-data) applies equally well to Core and ORM.
* [Data Manipulation with the ORM](https://docs.sqlalchemy.org/en/20/tutorial/orm_data_manipulation.html#tutorial-orm-data-manipulation) covers the persistence framework of the ORM; basically the ORM-centric ways to insert, update and delete, as well as how to handle transactions.
* [Worknig with ORM Related Objects](https://docs.sqlalchemy.org/en/20/tutorial/orm_related_objects.html#tutorial-orm-related-objects) introduces the concept of the [relationship()](https://docs.sqlalchemy.org/en/20/orm/relationship_api.html#sqlalchemy.orm.relationship) construct and provides a brief overview of how it's used, with links to deeper documentation.
* [Further Reading](https://docs.sqlalchemy.org/en/20/tutorial/further_reading.html#tutorial-further-reading) lists a series of major top-level documentation sections which fully document the concepts introduced in this tutorial.

In [1]:
import sqlalchemy
print(sqlalchemy.__version__)

2.0.45


# Establishing Connectivity - the Engine

## Welcome ORM and Core readers alike!

Every SQLAlchemy application that connects to a database needs to use an [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine). This short section is for everyone.

The start of any SQLAlchemy application is an object called the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine). This object acts as a central source of connections to a particular database, providing both a factor as well as a holding space called a [connection pool](https://docs.sqlalchemy.org/en/20/core/pooling.html) for these database connections. The engine is typically a global object created just once for a particular database server, and is configured using a URL string which will describe how it should connect to the datbase host or backend.

For this tutorial we will use SQLite database. The [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) is created by using the [create_engine()](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine) function:

In [2]:
from sqlalchemy import create_engine
dbPath = 'tutorial.db'
engine = create_engine('sqlite+pysqlite:///%s' % dbPath, echo=True)

To use an in-memory-only SQLite database, which is an easy way to test things without needing to have an actual pre-existing database set up:

```python
from sqlalchemy import create_engine
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)
```

The main argument to [create_engine](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine) is a string URL, above passed as the string `"sqlite_pysqlite:///tutorial.db"`. This string indicates to the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) three important facts:

1. What kind of database are we communicating with? This is the `sqlite` portion above, which links in SQLAlchemy to an object known as the [dialect](https://docs.sqlalchemy.org/en/20/glossary.html#term-dialect).
2. What [DBAPI](https://docs.sqlalchemy.org/en/20/glossary.html#term-DBAPI) are we using? The Python [DBAPI](https://docs.sqlalchemy.org/en/20/glossary.html#term-DBAPI) is a third party driver that SQLAlchemy uses to interact with a particular database. In this case, we're using the name `pysqlite`, which in modern Python use is the [sqlite3](https://docs.python.org/library/sqlite3.html) standard library interface for SQLite. It omitted, SQLAlchemy will use a default [DBAPI](https://docs.sqlalchemy.org/en/20/glossary.html#term-DBAPI) for the particular database selected.
3. How do we locate the database? In this case, our URL could include the phrase `/:memory:`, which is an indicator to the `sqlite3` module that we will be using an **in-memory-only** database. This kind of database is perfect for experimenting as it does not require any server nor does it need to create new files.

In [3]:
'sqlite+pysqlite:///%s' % dbPath

'sqlite+pysqlite:///tutorial.db'

## Lazy Connecting

The [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine), when first returned by [create_engine()](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine), has not actually tried to connect to the database yet; that happens only the first time it is asked to perform a task against the database. This is a software design pattern known as [lazy initialization](https://docs.sqlalchemy.org/en/20/glossary.html#term-lazy-initialization).

We have also specified a parameter [create_engine.echo](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine.params.echo), which will instruct the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) to log all of the SQL it emits to a Python logger that will write to standard out. This flag is a shorthand way of setting up [Python logging more formally](https://docs.sqlalchemy.org/en/20/core/engines.html#dbengine-logging) and is useful for experimentation in scripts. Many of the SQL examples will include this SQL logging output beneath a [ SQL ] link that when clicked, will reveal the full SQL interaction.

# Working with Transactions and the DBAPI

With the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) object ready to go, we can dive into the basic operation of an [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) and its primary endpoints, the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) and [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result). We'll also introduce the ORM's [facade](https://docs.sqlalchemy.org/en/20/glossary.html#term-facade) for these objects, known as the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session)


## Note to ORM readers

When using the ORM, the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) is managed by the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session). The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) in modern SQALchemy emphasizes a transactional and SQL execution pattern that is largely identical to that of the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) discussed below, so while this subsection is Core-centric, all of the concepts here are relevant to ORM use as well and is recommended for all ORM learners. The execution pattern used by the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) will be compared to the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) at the end of this section.


As we have yet to introduce the SQLAlchemy Expression Language that is the primary feature of SQLAlchemy, we'll use a simple construct within this package called the [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text) construct to write SQL statements, as **textual SQL**. Rest assured that textual SQL is the exception rather than the rule in day-to-day SQLAlchemy use, but it's always available.


## Getting a Connection

The purpose of the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) is to connect to the database by providing a [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection). When working with the Core directly, the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) object is how all interaction with the database is done. Because the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) creates an open resource against the database, we want to limit our use of this object to a specific context. The best way to do that is with a Python context manager, also known as [the with statement](https://docs.python.org/3/reference/compound_stmts.html#with). Below we use a textual SQL statement to show "Hello World". Textual SQL is created with a construct called [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text) which we'll discuss in more detail later.

In [4]:
from sqlalchemy import text
with engine.connect() as conn:
    result = conn.execute(text("SELECT 'hello world'"))
    print(result.all())

2026-01-21 12:46:59,627 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 12:46:59,628 INFO sqlalchemy.engine.Engine SELECT 'hello world'
2026-01-21 12:46:59,630 INFO sqlalchemy.engine.Engine [generated in 0.00279s] ()
[('hello world',)]
2026-01-21 12:46:59,633 INFO sqlalchemy.engine.Engine ROLLBACK


In the example above, the context manager creates a database connectoin and executes the operation in a transaction. the default behavior of teh Python DBAPI is that a transaction is always in progress; when the connection is [released](https://docs.sqlalchemy.org/en/20/glossary.html#term-released), a ROLLBACK is emitted to end the transactoin. The transaction is **not committed automatically**; if we want to commit data we need to call [Connection.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) as we'll see in the next section.


### Tip

"autocommit" mode is available for special cases. The section [Setting Transaction Isolation Levels Including DBAPI Autocommit](https://docs.sqlalchemy.org/en/20/core/connections.html#dbapi-autocommit) discusses this.


The result of our SELECT was returned in an object called [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) that will be discussed later. For the moment we'll add that it's best to use this object within the "connect" block, and not ot use it outside of the scope of our connection.


# Committing Changes

We just learned that the DBAPI connection doesn't commit automatically. What if we want to commit some data? We can change our example above to create a table, insert some data and then commit the transaction using the [Connection.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) method, **inside** the block where we have the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) object:

In [5]:
# "commit as you go"
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-21 12:46:59,651 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 12:46:59,654 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-21 12:46:59,656 INFO sqlalchemy.engine.Engine [generated in 0.00487s] ()


2026-01-21 12:46:59,657 INFO sqlalchemy.engine.Engine ROLLBACK


OperationalError: (sqlite3.OperationalError) table some_table already exists
[SQL: CREATE TABLE some_table (x int, y int)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Above, we execute two SQL statements, a "CREATE TABLE" statement and an "INSERT" statement that's parametrized (we discuss the parametrization syntax later in [Sending Multiple Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-multiple-parameters)). To commit the work we've done in our block, we call the [Connectoin.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) method which commits the transaction. After this, we can continue to run more SQL statements and call [Connection.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) again for those statements. SQLAlchemy refers to this style as **commit as you go**.


There's also another style to commit data. We can declare our "connect" block to be a transactoin block up front. to do this, we use the [Engine.begin()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine.begin) method to get the connection, rather than the [Engine.connect()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine.connect) method. This method will manage the scope of the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) and also enclose everything inside of a transaction with either a COMMIT at the end if the block was successful, or a ROLLBACK if an exception was raised. This style is known as **begin once**:

In [ ]:
# "begin once"
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-21 10:32:21,015 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 10:32:21,017 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-21 10:32:21,018 INFO sqlalchemy.engine.Engine [cached since 1758s ago] [(6, 8), (9, 10)]
2026-01-21 10:32:21,023 INFO sqlalchemy.engine.Engine COMMIT


You should mostly prefer the "begin once" style because it's shorter and shows the intention of the entire block up front. However, in this tutorial we'll use the "commit as you go" style as its more flexible for demonstration purposes.

## What's "BEGIN (implicit)"?

You might have noticed the log line "BEGIN (implicit)" at the start of a transaction block. "implicit" here means that SQLAlchemy **did not actually send any command** to the database; it just considers this to be the start of the DBAPI's implicit transaction. You can register [event hooks](https://docs.sqlalchemy.org/en/20/core/events.html#core-sql-events) to intercept this event, for example.


[DDL](https://docs.sqlalchemy.org/en/20/glossary.html#term-DDL) refers to the subset of SQL that instructs the database to create, modify, or remove schema-level constructs such as tables. DDL such as "CREATE TABLE" should be in a transaction block that ends with COMMIT, as many databases uses transactional DDL such that the schema changes don't take place until the transaction is committed. However, as we'll see later, we usually let SQLAlchemy run DDL sequences for us as part of a higher level operatoin where we don't generally need to worry about COMMIT.



## Basics of Statement Execution

We have seen a few examples that run SQL statements against a database, making use of a method called [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute), in conjunction with an object called [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text), and returning an object called [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result). In this section we'll illustrate more closely the mechanics and interactions of these components.


Most of the content of this section applies equally well to modern ORM use when using the [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) method, which works very similarly to that of [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute), including that ORM result rows are delivered using the same [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) interface used by Core.


## Fetching Rows

We'll first illustrate the [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) object more closely by making use of the rows we've inserted previously, running a textual SELECT statement on the table we've created:

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y from some_table"))
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-21 10:44:49,709 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 10:44:49,712 INFO sqlalchemy.engine.Engine SELECT x, y from some_table
2026-01-21 10:44:49,713 INFO sqlalchemy.engine.Engine [generated in 0.00354s] ()
x: 1, y: 1
x: 2, y: 4
x: 6, y: 8
x: 9, y: 10
2026-01-21 10:44:49,718 INFO sqlalchemy.engine.Engine ROLLBACK


Above, the "SELECT" string we executed selected all ows from our table. The object returned is called [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) and represents an iterable object of the result rows.

[Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) has lots of methods for fetching and transforming rows, such as the [Result.all()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result.all) method illustrated previously, which returns a list of all [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects. It also implements the Python iterator interface so that we can iterate over the collection of [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects directly.

The [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects themselves are intended to act like Python [named tuples](https://docs.python.org/3/library/collections.html#collections.namedtuple)

Below we illustrate a variety of ways to acccess rows

* **Tuple Assignment** - This is the most Python-idiomataic style, which is to assign variables to each row positionally as they are are received:

```python
result = conn.execute(text("select x, y from some_table"))

for x, y in result:
    ...
```


* **Integer Index** - Tuples are Python sequences, so regular integer access is available too:

```python
result = conn.execute(text("select x, y from some_table"))

for row in result:
    x = row[0]
```


* **Attribute Name** - As these are Python named tuples, the tuples have dynamic attribute names matching the names of each column. These names are normally the names that the SQL statement assigns to the columns in each row. While they are usually fairly predictable and can also be controlled by labels, in less defined cases they may be subject to database-specific behaviors:

```python
result = conn.execute(text("select x, y from some_table"))

for row in result:
    y = row.y

    # illustate use with Python f-strings
    print(f"Row: {row.x} {y}")
```


* **Mapping Access** - To receive rows as Python **mapping** objects, which is essentially a read-only version of Python's interface to the common `dict` object, the [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) may be **transformed** into a [MappingResult](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.MappingResult) object using the [Result.mappings()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result.mappings) modifier; this is a result object that yields dictionary-like [RowMapping](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.RowMapping) objects rather than [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects:

```python
result = conn.execute(text("select x, y from some_table"))

for dict_row in result.mappings():
    x = dict_row["x"]
    y = dict_row["y"]
```

## Sending Parameters

SQL statements are usually accompanied by data that is to be passed with the statement itself, as we saw in the INSERT example previously. The [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) method therefore also accepts parameters, which are known as [bound parameters](https://docs.sqlalchemy.org/en/20/glossary.html#term-bound-parameters). A rudimentary example might be if we wanted to limit our SELECT statement only to rows that meet a certain criteria, such as rows where the "y" value are greater than a certain value that is passed in to a function.

In order to achieve this such that the SQL statement can remain fixed and that the driver can properly sanitize the vlaues, we add a WHERE criteria to our statement that names a new parameter called "y", the [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text) construct accepts these using a colon format ":y". The actual value for ":y" is then passed as the second argument to [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) in the form of a dictionary:

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table WHERE y > :y"), {"y": 2})
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-21 11:29:41,890 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 11:29:41,892 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ?
2026-01-21 11:29:41,892 INFO sqlalchemy.engine.Engine [generated in 0.00201s] (2,)
x: 2, y: 4
x: 6, y: 8
x: 9, y: 10
2026-01-21 11:29:41,894 INFO sqlalchemy.engine.Engine ROLLBACK


In the logged SQL output, we can see that the bound parameter :y was converted into a question mark when it was sent to the SQLite database. This is because the SQLite database driver uses a format called "qmark parameter style", which is one of six different formats allowed by the DBAPI specification. SQLAlchemy abstracts these formats into just one, which is the "named" format with a colon.


### Always use bound parameters

As mentioned at the beginning of this section, textual SQL is not the usual way we work with SQLAlchemy. However, when using textual SQL, a Python literal value, even non-strings like integers or dates, should **never be stringified into SQL string directly**: a parameter should **always** be used. This ismost famously known as how to avoid SQL injection attacks when the data is untrusted. However, it also allows the SQLAlchemy dialects and/or DBAPI to correctly handle the incoming input for the backend. Outside of plain textual SQL use cases, SQlAlchemy's Core Expression API otherwise ensures that Python literal values are passed as bound parameters where appropriate. 

## Sending Multiple Parameters

In the example at [Committing Changes](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-committing-data), we executed an INSERT statement where it appeared that we were able to INSERT multiple rows into the database at once. For [DML](https://docs.sqlalchemy.org/en/20/glossary.html#term-DML) statements such as "INSERT", "UPDATE", and "DELETE", we can send **multiple parameter sets** to the [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) method by passing a list of dictionaries instead of a single dictionary, which indicates that the single SQL statement should be invoked multiple times, once for each parameter set. This style of execution is known as [executemany](https://docs.sqlalchemy.org/en/20/glossary.html#term-executemany):

In [ ]:
with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}],
    )
    conn.commit()

2026-01-21 11:44:29,033 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 11:44:29,035 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-21 11:44:29,036 INFO sqlalchemy.engine.Engine [cached since 6086s ago] [(11, 12), (13, 14)]
2026-01-21 11:44:29,038 INFO sqlalchemy.engine.Engine COMMIT


The above operation is equivalent to running the given INSERT statement once for eahc parameter set, except that the operation will be optimized for better performance across many rows.

A key behavioral difference between "execute" and "executemany" is that the latter doesn't support returning results of rows, even if the statement includes the RETURNING clause. The one exception to this is when using a Core [insert()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.insert) construct, introduced later in this tutorial at [Using INSERT Statements](https://docs.sqlalchemy.org/en/20/tutorial/data_insert.html#tutorial-core-insert), which also indicates RETURNING using the [Insert.returning()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.returning) method. In that case, SQLAlchemy makes use of special logic to reorganize the INSERT statement so that it can be invoked for many rows while stil support RETURNING.

### See Also

[executemany](https://docs.sqlalchemy.org/en/20/glossary.html#term-executemany) in the [Glossary](https://docs.sqlalchemy.org/en/20/glossary.html) describes the DBAPI-level [cursor.executemany()](https://peps.python.org/pep-0249/#executemany) method that's used for most "executemany" executions.  ["Insert Many Values" Behavior for INSERT statements](https://docs.sqlalchemy.org/en/20/core/connections.html#engine-insertmanyvalues) - in [Working with Engines and Connections](https://docs.sqlalchemy.org/en/20/core/connections.html), describes the specialized logic used by [Insert.returning()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.returning) to deliver result sets with "executemany" executions.

## Executing with an ORM Session

As mentioned previously, most of the patterns and examples above apply to use with the ORM as well, so here we will introduce this usage so that as the tutorial proceeds, we will be able to illustrate each pattern in terms of Core and ORM uses together.

The fundamental transactional / database interactive object when using the ORM is called the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session). In modern SQLAlchemy, this object is used in a manner very similar to that of the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection), and in fact as the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) is used, it refers to a [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) internally, which is uses to emit SQL.


When the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) is used with non-ORM constucts, it passes through the SQL statements we give it and does not generally do things much differently from how the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) does directly, so we can illustrate it here in terms of the simple textual SQL operations we've already learned.

The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) has a few different creational patterns, but here we will illustrate the most basic one that tracks exactly with how the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) is used which is to construct it within a context manager.

https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html

In [6]:
from sqlalchemy.orm import Session

stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-21 12:48:57,621 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 12:48:57,624 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-21 12:48:57,625 INFO sqlalchemy.engine.Engine [generated in 0.00132s] (6,)
x: 6 y: 8
x: 9 y: 10
x: 11 y: 12
x: 13 y: 14
2026-01-21 12:48:57,629 INFO sqlalchemy.engine.Engine ROLLBACK


The example above can be compared to the example in the preceding section in [Sending Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-sending-parameters) - we diretly replace the call to `with engine.connect() as conn` with `with Session(engine) as session`, and then make use of the [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) just like we do with the [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute)

Also, like the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection), the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) features "commit as you go" behavior using the [Session.commit()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.commit) method, illustrated below using a textual UPDATE statement to alter some of our data:

In [7]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}],
    )
    session.commit()

2026-01-21 13:36:58,822 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-21 13:36:58,825 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-21 13:36:58,827 INFO sqlalchemy.engine.Engine [generated in 0.00227s] [(11, 9), (15, 13)]
2026-01-21 13:36:58,830 INFO sqlalchemy.engine.Engine COMMIT


Above, we invoked an UPDATE statement using the bound-parameter, "executemany" style of execution introducted at [Sending Multiple Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-multiple-parameters), ending the block with a "commit as you go" commit.


### Tip

The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) doesn't actually hold onto the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) object after it ends the transaction. It gets a new [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) from the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) the next time it needs to execute SQL against the database.

The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) obviously has a lot more tricks up its sleeve than that, however understanding that it has a [Session.execute](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) method that's used the same way as [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) will get us started with the examples that follow later.